# Driver Code

In [1]:
from sklearn.model_selection import train_test_split

class Experiment:
    def __init__(self, X, y, train_size=0.8, test_size=0.1, val_size=0.1, split_type='train-val-test', print_stats=None):
        self.X = X
        self.y = y

        self.X_train = None
        self.y_train = None
        self.X_val = None
        self.y_val = None
        self.X_test = None
        self.y_test = None

        self.__split_data(train_size, test_size, val_size, split_type)
        if print_stats:
            self.__print_data_summary(train_size, test_size, val_size, split_type)

    def __split_data(self, train_size, test_size, val_size, split_type):
        if split_type == 'train-val-test':
            if not np.isclose(train_size + test_size + val_size, 1.0):
                raise ValueError("train_size, test_size, and val_size must sum to 1.0")

            if train_size == 1.0:
                self.X_train, self.y_train = self.X, self.y
                return

            X_train, X_temp, y_train, y_temp = train_test_split(
                self.X, self.y, train_size=train_size, random_state=42
            )

            self.X_train, self.y_train = X_train, y_train

            remaining_size = val_size + test_size
            if np.isclose(remaining_size, 0.0):
                return

            relative_test_size = test_size / remaining_size

            if np.isclose(relative_test_size, 1.0):
                self.X_test, self.y_test = X_temp, y_temp
            elif np.isclose(relative_test_size, 0.0):
                self.X_val, self.y_val = X_temp, y_temp
            else:
                self.X_val, self.X_test, self.y_val, self.y_test = train_test_split(
                    X_temp, y_temp, test_size=relative_test_size, random_state=42
                )

        elif split_type == 'train-test':
            if not np.isclose(train_size + test_size, 1.0):
                raise ValueError("train_size and test_size must sum to 1.0")

            if train_size == 1.0:
                self.X_train, self.y_train = self.X, self.y
            elif test_size == 1.0:
                self.X_test, self.y_test = self.X, self.y
            else:
                self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                    self.X, self.y, train_size=train_size, random_state=42
                )

        else:
            raise ValueError(f"Unknown split_type: {split_type}. Must be 'train-val-test' or 'train-test'.")

    def __print_data_summary(self, train_size, test_size, val_size, split_type):
        total_samples = len(self.X)
        train_samples = len(self.X_train) if self.X_train is not None else 0
        val_samples = len(self.X_val) if self.X_val is not None else 0
        test_samples = len(self.X_test) if self.X_test is not None else 0

        print(f"\n--- Experiment Data Initialized ---")
        print(f"Total Samples: {total_samples}")
        print(f"Split Type:    '{split_type}'")

        if split_type == 'train-val-test':
            print(f"  - Train: {train_size*100:>6.1f}% ({train_samples} samples)")
            print(f"  - Val:   {val_size*100:>6.1f}% ({val_samples} samples)")
            print(f"  - Test:  {test_size*100:>6.1f}% ({test_samples} samples)")
        elif split_type == 'train-test':
            print(f"  - Train: {train_size*100:>6.1f}% ({train_samples} samples)")
            print(f"  - Test:  {test_size*100:>6.1f}% ({test_samples} samples)")

        print(f"Total in splits: {train_samples + val_samples + test_samples}")
        print("-----------------------------------\n")


In [ ]:
import time
from scipy.spatial.distance import cdist
from cvxopt import matrix, solvers

from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from constants import OUTPUT_PATH

solvers.options['show_progress'] = False


class QuantileSVRExperiment(Experiment):
    def __init__(self, X, y, satellite, train_size=0.8, test_size=0.1, val_size=0.1,
                 split_type='train-val-test', print_stats=None, type='censored'):
        super().__init__(X, y, train_size, test_size, val_size, split_type, print_stats)
        self.satellite = satellite
        self.results_path = OUTPUT_PATH / f"qsvr_pi_estimation_{type}"
        os.makedirs(self.results_path, exist_ok=True)

        self.__scale_data()

    def __scale_data(self):
        self.x_scaler = MinMaxScaler()
        self.y_scaler = MinMaxScaler()

        self.X_train_scaled = self.x_scaler.fit_transform(self.X_train)
        self.X_val_scaled = self.x_scaler.transform(self.X_val)
        self.X_test_scaled = self.x_scaler.transform(self.X_test)

        self.y_train = self.y_train.reshape(-1, 1)
        self.y_val = self.y_val.reshape(-1, 1)
        self.y_test = self.y_test.reshape(-1, 1)

        self.y_train_scaled = self.y_scaler.fit_transform(self.y_train)
        self.y_val_scaled = self.y_scaler.transform(self.y_val)
        self.y_test_scaled = self.y_scaler.transform(self.y_test)

        self.y_train = self.y_train.ravel()
        self.y_val = self.y_val.ravel()
        self.y_test = self.y_test.ravel()
        self.y_train_scaled = self.y_train_scaled.ravel()
        self.y_val_scaled = self.y_val_scaled.ravel()
        self.y_test_scaled = self.y_test_scaled.ravel()

    @staticmethod
    def kernelfun(X, kerfPara, Y=None):
        if Y is None:
            Y = X
        if kerfPara['type'] == 'rbf':
            gamma = kerfPara['pars']
            sqdist = cdist(X, Y, 'sqeuclidean')
            return np.exp(-gamma * sqdist)
        else:
            raise ValueError("Unknown kernel function")

    @staticmethod
    def svtol(C):
        return 1e-5

    @staticmethod
    def nobias(kerfType):
        return 0

    @staticmethod
    def fit_quantile_svr(X, Y, kerfPara, C, tau, eps1=0.0):
        epsilon = QuantileSVRExperiment.svtol(C)
        n = X.shape[0]
        H = QuantileSVRExperiment.kernelfun(X, kerfPara)

        Hb_top = np.hstack([H, -H])
        Hb_bottom = np.hstack([-H, H])
        Hb = np.vstack([Hb_top, Hb_bottom])

        Y_col = Y.reshape(-1, 1)
        c_part1 = ((1 - tau) * eps1 * np.ones((n, 1)) - Y_col)
        c_part2 = (tau * eps1 * np.ones((n, 1)) + Y_col)
        c_vec = np.vstack([c_part1, c_part2]).flatten()

        vlb = np.zeros(2 * n)
        vub = np.concatenate([tau * C * np.ones(n), (1 - tau) * C * np.ones(n)])

        P = matrix(Hb)
        q = matrix(c_vec)
        I = np.eye(2 * n)
        G1 = -I
        h1 = np.zeros(2 * n)
        G2 = I
        h2 = vub
        G = matrix(np.vstack([G1, G2]))
        h = matrix(np.hstack([h1, h2]))

        sol = solvers.qp(P, q, G, h)

        alpha = np.array(sol['x']).flatten()
        alpha1 = alpha[:n]
        beta1 = alpha[n:2*n]
        beta = alpha1 - beta1
        bias = 0

        return beta, bias, H

    @staticmethod
    def predict_quantile_svr(X_train, X_predict, kerfPara, beta, bias):
        H_test = QuantileSVRExperiment.kernelfun(X_predict, kerfPara, X_train)
        return H_test.dot(beta) + bias

    def evaluate_model(self, y_true, y_pred_lower, y_pred_upper):
        y_true_flat = y_true.flatten()
        y_lower_flat = y_pred_lower.flatten()
        y_upper_flat = y_pred_upper.flatten()

        covered = np.sum((y_true_flat >= y_lower_flat) & (y_true_flat <= y_upper_flat))
        picp = covered / len(y_true_flat)
        mpiw = np.mean(y_upper_flat - y_lower_flat)

        return {
            'PICP': float(picp),
            'MPIW': float(mpiw)
        }

    def plot_prediction_interval(self, y_pred_lower_test, y_pred_upper_test,
                                 y_pred_lower_val, y_pred_upper_val, model_param_string):
        indices = range(len(self.y_test))

        test_eval = self.evaluate_model(self.y_test, y_pred_lower_test, y_pred_upper_test)
        val_eval  = self.evaluate_model(self.y_val,  y_pred_lower_val,  y_pred_upper_val)

        fig, ax = plt.subplots(figsize=(14, 7))
        ax.plot(indices, self.y_test, 'o', color='blue', label='Actual Soil Moisture (Test Set)', markersize=4)
        ax.plot(indices, y_pred_lower_test, color='red',    linestyle='--', label='Lower Bound')
        ax.plot(indices, y_pred_upper_test, color='orange', linestyle='--', label='Upper Bound')
        ax.fill_between(indices, y_pred_lower_test, y_pred_upper_test, color='gray', alpha=0.2, label='95% Prediction Interval')

        metrics_text = (f"Test  | PICP: {test_eval['PICP']*100:5.2f}% | MPIW: {test_eval['MPIW']:.4f}"
                        f"    Valid | PICP: {val_eval['PICP']*100:5.2f}% | MPIW: {val_eval['MPIW']:.4f}")

        plot_dir = self.results_path / "plots"
        os.makedirs(plot_dir, exist_ok=True)

        ax.set_xlabel('Sample Index')
        ax.set_ylabel('Soil Moisture (%)')
        ax.set_title(f'{self.satellite}: {model_param_string}\nQ-SVR Prediction Interval')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        fig.text(0.5, -0.02, metrics_text, ha='center', va='top',
                 fontsize=10, fontfamily='monospace',
                 bbox=dict(boxstyle="round,pad=0.4", facecolor="lightyellow", alpha=0.9))
        plot_save_path = plot_dir / f"{self.satellite}_{model_param_string}.png"
        plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')
        plt.close()

    def run_experiment(self, C_value, gamma, q_lower=0.025, q_upper=0.975, eps1=0.0):
        model_param_string = f"C={C_value}_gamma={gamma:.5g}"
        kerfPara = {'type': 'rbf', 'pars': gamma}

        print(f"  Training lower quantile ({q_lower})...")
        t0 = time.time()
        beta_lower, bias_lower, _ = self.fit_quantile_svr(
            self.X_train_scaled, self.y_train, kerfPara, C_value, q_lower, eps1
        )
        print(f"  Done in {time.time()-t0:.1f}s")

        print(f"  Training upper quantile ({q_upper})...")
        t0 = time.time()
        beta_upper, bias_upper, _ = self.fit_quantile_svr(
            self.X_train_scaled, self.y_train, kerfPara, C_value, q_upper, eps1
        )
        print(f"  Done in {time.time()-t0:.1f}s")

        y_preds_lower_val  = self.predict_quantile_svr(self.X_train_scaled, self.X_val_scaled,  kerfPara, beta_lower, bias_lower)
        y_preds_upper_val  = self.predict_quantile_svr(self.X_train_scaled, self.X_val_scaled,  kerfPara, beta_upper, bias_upper)
        y_preds_lower_test = self.predict_quantile_svr(self.X_train_scaled, self.X_test_scaled, kerfPara, beta_lower, bias_lower)
        y_preds_upper_test = self.predict_quantile_svr(self.X_train_scaled, self.X_test_scaled, kerfPara, beta_upper, bias_upper)

        self.plot_prediction_interval(
            y_preds_lower_test, y_preds_upper_test,
            y_preds_lower_val,  y_preds_upper_val,
            model_param_string
        )

        results_val  = self.evaluate_model(self.y_val,  y_preds_lower_val,  y_preds_upper_val)
        results_test = self.evaluate_model(self.y_test, y_preds_lower_test, y_preds_upper_test)

        results = {
            "params": {"C": C_value, "gamma": gamma, "q_lower": q_lower, "q_upper": q_upper},
            "val":  results_val,
            "test": results_test
        }
        print(f"  PICP test={results_test['PICP']*100:.2f}%  MPIW={results_test['MPIW']:.4f}")
        return results

# Experiment Code

In [3]:
from constants import DATA_PATH
import pandas as pd

eos = pd.read_csv(DATA_PATH / "eos-04-processed.csv")
sentinel = pd.read_csv(DATA_PATH / "sentinel-1-processed.csv")

In [4]:
sentinel = sentinel[sentinel['SM1 (%)'] != 50]
eos = eos[eos['SM1 (%)'] != 50]

In [5]:
from constants import X_cols_eos, X_cols_sentinel

y_col = ['SM1 (%)']

X_sentinel = sentinel[X_cols_sentinel].values
X_eos = eos[X_cols_eos].values

y_sentinel = sentinel[y_col].values
y_eos = eos[y_col].values

In [ ]:
C_VALUES     = [2**i for i in range(1, 10, 2)]   # [2, 8, 32, 128, 512]
GAMMA_VALUES = [2**i for i in range(-15, 16)]     # 2^-15 … 2^15 (31 values)

print(f"C values    ({len(C_VALUES)}):  {C_VALUES}")
print(f"Gamma values ({len(GAMMA_VALUES)}): 2^-15 to 2^15")
print(f"Total combos per satellite: {len(C_VALUES) * len(GAMMA_VALUES)}")

In [ ]:
sentinel_exp = QuantileSVRExperiment(
    X=X_sentinel, y=y_sentinel,
    satellite="Sentinel-1", print_stats=False, type="uncensored"
)

all_results = []
total_combos = len(C_VALUES) * len(GAMMA_VALUES)

for c_idx, C_val in enumerate(C_VALUES):
    for g_idx, gamma in enumerate(GAMMA_VALUES):
        run_num = c_idx * len(GAMMA_VALUES) + g_idx + 1
        print(f"\n[{run_num}/{total_combos}] C=2^{int(np.log2(C_val))} ({C_val}), "
              f"Gamma=2^{int(np.log2(gamma))} ({gamma:.5g})")
        try:
            results = sentinel_exp.run_experiment(C_value=C_val, gamma=gamma)
            all_results.append(results)
        except Exception as e:
            print(f"  ERROR: {e}")
            all_results.append({
                "params": {"C": C_val, "gamma": gamma},
                "val":  {"PICP": None, "MPIW": None},
                "test": {"PICP": None, "MPIW": None},
                "error": str(e)
            })

In [ ]:
results_df = pd.json_normalize(all_results)
results_df.columns = results_df.columns.str.replace('params.', 'param_', regex=False)

try:
    results_df['param_gamma_str'] = '2^' + np.log2(results_df['param_gamma']).astype(int).astype(str)
    results_df['param_C_str']     = '2^' + np.log2(results_df['param_C']).astype(int).astype(str)
except Exception as e:
    print(f"Warning: {e}")

summary_path = sentinel_exp.results_path / "tuning_summary_2D.csv"
results_df.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")

acceptable = results_df[
    (results_df['test.PICP'] > 0.90) & (results_df['test.PICP'] <= 0.97)
].copy()

if not acceptable.empty:
    acceptable = acceptable.sort_values(by="test.MPIW", ascending=True)
    print("\n--- Best models: 90-97% PICP, lowest MPIW ---")
    print(acceptable[['param_C_str', 'param_gamma_str', 'test.PICP', 'test.MPIW']].head(10).to_string(index=False))
else:
    print("\nNo models achieved 90-97% PICP.")

print("\n--- Top 10 by PICP then MPIW ---")
print(results_df.sort_values(["test.PICP", "test.MPIW"], ascending=[False, True])
      [['param_C_str', 'param_gamma_str', 'test.PICP', 'test.MPIW']].head(10).to_string(index=False))

In [9]:
results_df.sort_values(by="test.PICP", ascending=False)[['param_gamma_str', 'test.PICP', 'test.MPIW']]

,param_gamma_str,test.PICP,test.MPIW
0,2^-15,0.956204,42.204059
1,2^-14,0.956204,42.208114
2,2^-13,0.956204,42.197456
3,2^-12,0.956204,42.128269
4,2^-11,0.956204,42.006756
5,2^-10,0.948905,41.780371
6,2^-9,0.948905,41.176675
7,2^-8,0.941606,40.082845
8,2^-7,0.941606,39.278010
9,2^-6,0.941606,38.040404


## EOS

In [ ]:
eos_exp = QuantileSVRExperiment(
    X=X_eos, y=y_eos,
    satellite="EOS-04", print_stats=False, type="uncensored"
)

all_results = []
total_combos = len(C_VALUES) * len(GAMMA_VALUES)

for c_idx, C_val in enumerate(C_VALUES):
    for g_idx, gamma in enumerate(GAMMA_VALUES):
        run_num = c_idx * len(GAMMA_VALUES) + g_idx + 1
        print(f"\n[{run_num}/{total_combos}] C=2^{int(np.log2(C_val))} ({C_val}), "
              f"Gamma=2^{int(np.log2(gamma))} ({gamma:.5g})")
        try:
            results = eos_exp.run_experiment(C_value=C_val, gamma=gamma)
            all_results.append(results)
        except Exception as e:
            print(f"  ERROR: {e}")
            all_results.append({
                "params": {"C": C_val, "gamma": gamma},
                "val":  {"PICP": None, "MPIW": None},
                "test": {"PICP": None, "MPIW": None},
                "error": str(e)
            })

In [ ]:
results_df = pd.json_normalize(all_results)
results_df.columns = results_df.columns.str.replace('params.', 'param_', regex=False)

try:
    results_df['param_gamma_str'] = '2^' + np.log2(results_df['param_gamma']).astype(int).astype(str)
    results_df['param_C_str']     = '2^' + np.log2(results_df['param_C']).astype(int).astype(str)
except Exception as e:
    print(f"Warning: {e}")

summary_path = eos_exp.results_path / "eos-04-tuning_summary_2D.csv"
results_df.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")

acceptable = results_df[
    (results_df['test.PICP'] > 0.90) & (results_df['test.PICP'] <= 0.97)
].copy()

if not acceptable.empty:
    acceptable = acceptable.sort_values(by="test.MPIW", ascending=True)
    print("\n--- Best models: 90-97% PICP, lowest MPIW ---")
    print(acceptable[['param_C_str', 'param_gamma_str', 'test.PICP', 'test.MPIW']].head(10).to_string(index=False))
else:
    print("\nNo models achieved 90-97% PICP.")

print("\n--- Top 10 by PICP then MPIW ---")
print(results_df.sort_values(["test.PICP", "test.MPIW"], ascending=[False, True])
      [['param_C_str', 'param_gamma_str', 'test.PICP', 'test.MPIW']].head(10).to_string(index=False))

In [12]:
results_df.sort_values(by="test.PICP", ascending=False)[['param_gamma_str', 'test.PICP', 'test.MPIW']]

,param_gamma_str,test.PICP,test.MPIW
8,2^-7,0.972067,39.154698
7,2^-8,0.966480,40.086365
9,2^-6,0.966480,38.401812
3,2^-12,0.960894,41.805277
0,2^-15,0.960894,41.975656
1,2^-14,0.960894,41.951312
2,2^-13,0.960894,41.902628
5,2^-10,0.960894,41.309499
6,2^-9,0.960894,41.057071
4,2^-11,0.955307,41.702243
